In [1]:
import pystac
from pathlib import Path
import pathlib
import copclib as copc
import pymeepcl as mee
import pymeepcl.visuals as vis
import pyproj


import laspy
import rasterio as rio

from dataclasses import dataclass, field

import os
import json
import rasterio as rio
import pystac
from datetime import datetime, timezone
from shapely.geometry import Polygon, mapping

# Beispiel extensions außer projection wsl nicht wichtig
from pystac.extensions.eo import Band, EOExtension # in stac 1.1 Bands is common
from pystac.extensions.view import ViewExtension # dont know if applies
from pystac.extensions.projection import ProjectionExtension
from pystac.extensions.pointcloud import PointcloudExtension
from pystac.extensions.file import FileExtension
from pystac.extensions.raster import RasterExtension

from typing import Optional

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Catalog

catalog = pystac.Catalog(id='test-catalog', description='This catalog is a test run for pielach timeseries')

item = pystac.Item(id='local-image',
		geometry=footprint,
		bbox=bbox,
		datetime=datetime_utc,
		properties={})

catalog.add_item(item)

print(list(catalog.get_children()))
print(list(catalog.get_items()))

# Add collection to catalog
catalog.add_child(collection)

# Asset

item.add_asset(
	key='image',
	asset=pystac.Asset(
		href=img_path,
		media_type=pystac.MediaType.GEOTIFF)
	)

catalog.make_all_asset_hrefs_relative()

# Save

catalog.normalize_hrefs(os.path.join(tmp_dir.name, "stac"))
catalog.save(catalog_type=pystac.CatalogType.[type])

# Extensions / Common Metadata

eo = EOExtension.ext(item, add_if_missing=True)
eo.apply(bands=wv3_bands)


item.common_metadata.platform = "Maxar"
item.common_metadata.instruments = ["WorldView3"]
item.common_metadata.gsd = 0.3

# Collections

from shapely.geometry import shape

unioned_footprint = shape(footprint).union(shape(footprint2))
collection_bbox = list(unioned_footprint.bounds)
spatial_extent = pystac.SpatialExtent(bboxes=[collection_bbox])


collection_interval = sorted([collection_item.datetime, collection_item2.datetime])

temporal_extent = pystac.TemporalExtent(intervals=[collection_interval])

# need extent and license
collection_extent = pystac.Extent(spatial=spatial_extent, temporal=temporal_extent)

collection = pystac.Collection(
	id='wv3-images',
	description='Spacenet 5 images over Moscow',
	extent=collection_extent,
	license='CC-BY-SA-4.0'
	)
	
collection.add_items(List[Item])

In [ ]:
root_catalog = pystac.Catalog(id='test-catalog', description='This catalog is a test run for pielach timeseries')

In [ ]:
@dataclass
class pc_schema:
    name: str
    size: int
    type: str

@dataclass
class pc_metadata:
    pc_count: int
    pc_path: str
    pc_schemas: list[pc_schema] = field(default_factory=list)
    pc_encoding: str = "copc" # "LASzip" | "LAZ" | "EPT" | ...
    pc_type: str  = "lidar" # "lidar" | "eopc" | "radar" | "sonar" | "other"

In [ ]:
root = Path(r"C:\data\Bachelor\2023-02-08\pielach_2023-02-08_tiles")
pointcloud_paths = [x for x in root.iterdir()]

KIND_TO_TYPE = {
    "SignedInteger":   "signed",
    "UnsignedInteger": "unsigned",
    "FloatingPoint":   "floating",
    "BitField":        "unsigned",   # sub-byte flags; treat as unsigned
}

def extract_pc_metadata(path: pathlib.WindowsPath) -> pc_metadata:
    with laspy.open(str(path)) as f:
        header = f.header
        count = header.point_count
        schemas = []

        for d in header.point_format.dimensions:
            # STAC wants size in whole bytes; bitfields round up to 1.
            size_bytes = max(1, -(-d.num_bits // 8))
            schemas.append(pc_schema(d.name, size_bytes, KIND_TO_TYPE[d.kind.name]))

    return pc_metadata(
        count,
        str(path),
        schemas
    )

In [ ]:
pc_metadata_list = [extract_pc_metadata(str(f)) for f in pointcloud_paths]
for x in pc_metadata_list[0:5]: # print first 5 to confirm
    print(x)



In [ ]:
struct = mee.MeeStruct(str(pointcloud_paths[0]))
file = struct.files[0]
print(struct.bounds_las)
print(file.config.wkt)
with laspy.open(file.filepath) as f:
    crs = f.header.parse_crs()      # returns a pyproj.CRS (or None)
    print(crs.to_epsg())            # -> 25833
    print(crs.to_wkt()) 

    transformer = pyproj.Transformer.from_crs(f"epsg:{crs.to_epsg()}", "epsg:4326", always_xy=True)


In [ ]:
from shapely.geometry import Polygon, mapping
import datetime

def bbox_to_geometry(bbox):
    minx, miny, maxx, maxy = bbox
    return mapping(
        Polygon([
            (minx, miny),
            (minx, maxy),
            (maxx, maxy),
            (maxx, miny),
            (minx, miny)
        ])
    )

def create_pointcloud_item(path: pathlib.WindowsPath):
    # pymeepcl struct / file
    struct = mee.MeeStruct(str(path))
    file = struct.files[0]
    metadata = extract_pc_metadata(path)

    with laspy.open(file.filepath) as f:
        crs = f.header.parse_crs()
        transformer = pyproj.Transformer.from_crs(f"epsg:{crs.to_epsg()}", "epsg:4326", always_xy=True)


    minX, minY, _ = file.bounds_las_min
    maxX, maxY, _ = file.bounds_las_max
    lon_min, lat_min = transformer.transform(minX, minY)
    lon_max, lat_max = transformer.transform(maxX, maxY)
    bbox_wgs84 = [lon_min, lat_min, lon_max, lat_max]
    item = pystac.Item(
        id=str(path.name).replace(".copc.laz", ""),
        geometry=bbox_to_geometry(bbox_wgs84),
        bbox=bbox_wgs84,
        datetime=datetime.datetime(2023, 2, 8),
        properties={}
    )

    # Assets
    item.add_asset("pointcloud",
                   pystac.Asset(href=str(path), media_type="application/vnd.laszip+copc", roles=["data"]))
    # item.add_asset("DSM",
    #                pystac.Asset(href=asset_paths["dsm"], media_type="image/tiff; application=geotiff", roles=["data"]))
    # item.add_asset("thumbnail", pystac.Asset(href=asset_paths["thumb"], media_type="image/jpeg", roles=["thumbnail"],
    #                                          title="Overview thumbnail"))

    # Pointcloud Extension
    pce = PointcloudExtension.ext(item, add_if_missing=True)
    pce.count = metadata.pc_count
    pce.type = "lidar"
    pce.encoding = "copc"
    
    schemas = [
        {
            "name": schema.name,
            "type": schema.type,
            "size": schema.size,

        }
        for schema in metadata.pc_schemas
    ]
    item.properties["pc:schemas"] = schemas

    # Projection Extension
    proj = ProjectionExtension.ext(item, add_if_missing=True)
    proj.epsg = int(crs.to_epsg())

    return item

In [ ]:
items = [create_pointcloud_item(path) for path in pointcloud_paths]
items[0]

In [ ]:
collection = pystac.Collection(
        id="test",
        title="test title",
        description="still testing",
        license="proprietary",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[]]),
            temporal=pystac.TemporalExtent([[datetime.datetime(2023, 2, 8), datetime.datetime(2023, 2, 8)]])
        )
    )

for item in items:
    collection.add_item(item)

struct = mee.MeeStruct(r"C:\data\Bachelor\2023-02-08\pielach_2023-02-08_tiles")
struct.bounds_las

minX, minY, _ = struct.bounds_las[0]
maxX, maxY, _ = struct.bounds_las[1]
lon_min, lat_min = transformer.transform(minX, minY)
lon_max, lat_max = transformer.transform(maxX, maxY)

collection.extent.spatial = pystac.SpatialExtent([[lon_min, lat_min, lon_max, lat_max]])

In [ ]:
collection.normalize_hrefs(r"..\data\collection_test")
collection.validate_all()
collection.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)